In [ ]:
import smtplib
import imaplib
import email
from email.message import EmailMessage
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
from email.utils import make_msgid
from email.header import decode_header
from email.utils import parsedate_to_datetime
from imap_tools import MailBox, A

import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
from collections import defaultdict

import os
import csv

from itertools import zip_longest
import time
import random
from datetime import datetime
import pytz
import importlib
# from utils import send_gmail

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import Select
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.firefox.firefox_profile import FirefoxProfile
from urllib.parse import urlparse

from session_id import get_cookies


In [ ]:
def juntada_resposta_projudi(destinatario, url_recebimento, assunto, cookies, 
                             cookie_domain, link_base,
                             digitar, tempo_espera, data_resposta, hora_resposta):
    profile = FirefoxProfile()
    # Define o User-Agent (exatamente o mesmo que você usou no requests)
    profile.set_preference("general.useragent.override",
                           "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0"
                           )
    # Define o idioma do navegador (opcional)
    profile.set_preference("intl.accept_languages", "pt-BR,pt;q=0.9,en;q=0.8")
    # Cria opções e adiciona o perfil
    options = Options()
    options.profile = profile
            
    driver = webdriver.Firefox(service=Service(GeckoDriverManager().install()),options=options)
            # Define o tamanho da janela (parece mais com uso humano)
    driver.set_window_size(1280, 800)
            # Abre a página desejada
    driver.get(link_base)
    time.sleep(tempo_espera)
    for name, value in cookies.items():
        driver.add_cookie({
            'name': name,
            'value': value,
            'path': '/',           # geralmente '/'
            'domain': cookie_domain,  # importante: deve bater com o domínio acessado
            'secure': True         # define se o cookie é só para HTTPS
            })
    for cookie in driver.get_cookies():
        print(cookie)
            
    try:
        wait = WebDriverWait(driver, 20)
        driver.get(url_recebimento)
        driver.execute_script("window.scrollBy(0, 567);")

        # Preencher código da movimentação
        codigo_movimentacao = wait.until(EC.presence_of_element_located((By.ID, "seqCategoriaMovimentacao")))
        codigo_movimentacao.clear()
        codigo_movimentacao.send_keys("2011")
        time.sleep(tempo_espera)

        # Botão buscar movimentação
        botao = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, "btnBuscaMovimentacao"))
        )
        botao.click()
        time.sleep(tempo_espera)

        # Preencher observação
        campo_observacao = wait.until(EC.presence_of_element_located((By.ID, "observacao")))
        observacao = f"""RECEBIDO Ofício ref: {assunto} em {data_resposta} - {hora_resposta} hs, por: {destinatario}"""
        observacao = observacao.strip()
        observacao = re.sub(r'\s+', ' ', observacao)
        observacao = observacao[:1500]
        campo_observacao.clear()
        digitar(observacao, campo_observacao)


        driver.execute_script("window.scrollBy(0, 567);")

        # Clicar para concluir
        botao_concluir = wait.until(EC.element_to_be_clickable((By.ID, "Concluir")))
        botao_concluir.click()
        time.sleep(tempo_espera)

        #  Espera o alerta aparecer (até 10 segundos, ajustável)
        # alert = WebDriverWait(driver, 10).until(EC.alert_is_present())

        # Aceitar alerta
        alert = WebDriverWait(driver, 10).until(EC.alert_is_present())
        print(f"Alerta exibido: {alert.text}")
        alert.accept()
        
        print("✅ Juntada realizada com sucesso!")
        time.sleep(tempo_espera)
        driver.quit()
       
    
        return True
    except Exception as e:
        print(f"❌ Erro ao realizar juntada: {e}")
        return False

In [ ]:
#document.cookie.split('; ').reduce((acc, cookie) => {const [name, value] = cookie.split('=');acc[name] = value;return acc;}, {});
# pafonso - kvgm hvzm bxbz qibz; ouys uorp vqpr fqig
#user pafonso - pafonso.2vsj@gmail.com
# senha_app = 'ptsd daec xeyw dxgv'
# usuario = 'iguedes953@gmail.com'
cookies = get_cookies()
tempo_espera = random.uniform(1, 5)
senha_app = 'ouysuorpvqprfqig'
usuario = 'pafonso.2vsj@gmail.com'
IMAP_SERVER = 'imap.gmail.com'
NUM_MAX_EMAILS = 100
PADRAO_ASSUNTO = re.compile(r"2.*VSJ", re.IGNORECASE)


# Configurações do SMTP
SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 587
SMTP_USER = usuario
SMTP_PASSWORD = senha_app
REMETENTE = SMTP_USER
DESTINATARIO = 'pafonso-2vsj@tjba.jus.br'

def encaminhar_email_completo(msg):
    email = EmailMessage()

    # Assunto com prefixo "Enc:"
    email['Subject'] = f"Enc: {msg.subject or '(sem assunto)'}"
    email['From'] = REMETENTE
    email['To'] = DESTINATARIO

    # Corpo do e-mail com conteúdo original
    corpo_original = msg.text or msg.html or "(sem conteúdo)"

    corpo = f"""
Encaminhado automaticamente.

────────────────────────────────────
📨 Remetente original: {msg.from_}
📅 Data: {msg.date.strftime('%d/%m/%Y %H:%M')}
📄 Assunto original: {msg.subject}

📎 Anexos: {[anexo.filename for anexo in msg.attachments if anexo.filename]}

────────────────────────────────────

{corpo_original}
"""
    email.set_content(corpo)

    # Anexar todos os PDFs encontrados
    pdfs_encontrados = 0
    for anexo in msg.attachments:
        nome = anexo.filename or ''
        if nome.lower().endswith('.pdf'):
            email.add_attachment(anexo.payload, maintype='application', subtype='pdf', filename=nome)
            pdfs_encontrados += 1

    if pdfs_encontrados == 0:
        print(f"⚠️ Nenhum PDF encontrado em: {msg.subject}")
        return

    # Enviar via SMTP
    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as smtp:
            smtp.starttls()
            smtp.login(SMTP_USER, SMTP_PASSWORD)
            smtp.send_message(email)
        print(f"✅ E-mail '{msg.subject}' encaminhado com sucesso para {DESTINATARIO}")
    except Exception as e:
        print(f"❌ Erro ao encaminhar e-mail: {e}")

In [ ]:
# import csv
# import os

dados = []
path_csv = 'protocolo_email_projudi.csv'
msg_ids_registrados = set()

CAMPOS = [
    'processo', 'num_oficio', 'email_destino', 'status',
    'data_envio', 'hora_envio', 'registrar_envio',  'resposta', 'msg_id',
    'url_oficio', 'url_processo', 'url_recebimento', 'url_baixa', 'assunto',
]

# Cria arquivo se não existir
if not os.path.exists(path_csv):
    print(f"[INFO] Arquivo {path_csv} não existe. Criando novo.")
    with open(path_csv, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
        writer.writeheader()

# Carrega CSV e remove duplicados
chaves_vistas = set()  # chave composta para evitar duplicados
with open(path_csv, encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        # Evita linha de cabeçalho duplicado
        if str(row.get('processo', '')).strip().lower() == 'processo':
            continue

        # Normaliza campos
        for campo in CAMPOS:
            if campo not in row:
                row[campo] = ''
            elif row[campo] is None:
                row[campo] = ''
            else:
                row[campo] = str(row[campo]).strip()

        # Garantir que 'resposta' não fique vazia
        if not row.get('resposta', '').strip():
            row['resposta'] = ''

        # Cria chave única por assunto + data_envio + hora_envio
        chave = (
            (row.get('assunto', '').strip().lower() or '') + '_' +
            (row.get('data_envio', '').strip() or '') + '_' +
            (row.get('hora_envio', '').strip() or '')
        )
        msg_id = row.get('msg_id', '').strip()

        # Ignora duplicados
        if chave in chaves_vistas or (msg_id and msg_id in msg_ids_registrados):
            continue

        chaves_vistas.add(chave)
        if msg_id:
            msg_ids_registrados.add(msg_id)

        dados.append(row)

# Sobrescreve o CSV sem duplicados
with open(path_csv, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
    writer.writeheader()
    writer.writerows(dados)

print(f"[INFO] CSV atualizado sem duplicados. Total de registros: {len(dados)}")
print(f"🧾 {len(msg_ids_registrados)} mensagens já registradas.")

In [ ]:
# document.cookie.split('; ').reduce((acc, cookie) => {const [name, value] = cookie.split('=');acc[name] = value;return acc;}, {});
# Abre a página dos ofícios já logado

link_base = 'https://projudi.tjba.jus.br/projudi/'
oficios = 'listagens/CumprimentoCartorio?tipo=oficio&acao=expedidos'
url_oficios = link_base + oficios
# Faz a requisição já logado com requests
url = 'https://projudi.tjba.jus.br/projudi/listagens/CumprimentoCartorio?tipo=oficio'

parsed = urlparse(url)
cookie_domain = parsed.hostname 

print(parsed.hostname)  # ➜ 'projudi.tjba.jus.br'
print(parsed.path)


cookies = {
    'ADC_CONN_539B3595F4E':	"574CAA1662357EBBF5F94ED94FF66FE08C10454FD7424D566BCB2DD33BADE9781AE5CBFD4E50793B",
    'ADC_REQ_2E94AF76E7':	"17EB80EF2601FF13E17C4D41F54A5E701355460BBCC88D1E3722D8C48FB521953AE37A4688C180D2",
    'ADRUM':	"s~1773170981656&r~aHR0cHMlM0ElMkYlMkZwcm9qdWRpLnRqYmEuanVzLmJyJTJGcHJvanVkaSUyRg==",
    'JSESSIONID':	"B35B5F86AE9CED4135E5A05BE4E0EA6D.tomcat09-03",
    }

# Simula um navegador real

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    'Accept-Language': 'pt-BR,pt;q=0.9,en;q=0.8',
    'Referer': url_oficios,  # onde o clique teria ocorrido
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
}

response = requests.get(url_oficios, headers=headers, cookies=cookies)
# Analisa o HTML retornado
html = response.text
soup = BeautifulSoup(html, 'html.parser')
# print(soup.prettify()[:200])
print(url_oficios)

html = response.text
# 3. Faz o parser da página com os links
soup = BeautifulSoup(html, 'html.parser')
# 🔹 2. Encontra o <a> com o texto "última"
# link_ultima = soup.find('a', string=re.compile(r'ultima', re.IGNORECASE))

# 2. Encontra todos os links do tipo goToPage(N)
ultima_pagina = '94'
# links_paginas = soup.find_all('a', href=re.compile(r'goToPage\(\d+\)'))
# print(links_paginas[-1])
# if links_paginas[-1]:
#     match = re.search(r'goToPage\((\d+)\)', links_paginas[-1]['href'])
#     if match:
#         ultima_pagina = match.group(1)      # ➜ '89' (string)
#         # numero_int = int(numero_str)     # ➜ 89 (inteiro)

#         print("Como string:", ultima_pagina)
        # print("Como número:", numero_int)

# Simula o "clique" na última página
data_oficio = {
    'tipo': 'oficio',
    'acao': 'expedidos',
    'codTipoJustica': "2",
    'pagina': ultima_pagina,# <-- Este número veio de goToPage(89)
    'coluna' :'CumprimentoCartorio.CODCUMPRIMENTO',
    'ordem':"ASC"
}



# 1. Requisição como se fosse o clique no link
response = requests.post(url_oficios, headers=headers, cookies=cookies, data=data_oficio)
print(response.url)        # Verifica se foi redirecionado
print(response.status_code)
print(response.text[:500])  # Mostra o começo do HTML
# response = requests.get(url_oficios, headers=headers, cookies=cookies)
# print(response.status_code)
# print(response.text)  # ou .content para binário

html = response.text
# 3. Faz o parser da página com os links
soup = BeautifulSoup(html, 'html.parser')
link_ultima_pagina = soup.find('a', href=re.compile(r'goToPage\(\d+\)'))
print(link_ultima_pagina)
# limpo = limpar_html_para_email(html, 'html.parser')

In [ ]:

def registrar_encaminhamento_por_assunto(msg, path_csv=path_csv):
    agora = datetime.now()
    data_envio = agora.strftime('%d/%m/%Y')
    hora_envio = agora.strftime('%H:%M:%S')

    remetente = msg.from_
    assunto = msg.subject or "(sem assunto)"
    msg_id = getattr(msg, 'uid', '') or getattr(msg, 'message_id', '')  # pega UID ou message_id

   
     # Extrair o nome do PDF relevante
    nome_arquivo = ''
    for anexo in msg.attachments:
        nome = (anexo.filename or '').strip()
        if nome.lower().endswith('.pdf'):
            nome_arquivo = nome
            break
    linha = [remetente, assunto, data_envio, hora_envio, msg_id, nome_arquivo]
    novo_arquivo = not os.path.exists(path_csv)

    with open(path_csv, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter=';')
        if novo_arquivo:
            writer.writerow(['remetente', 'assunto', 'data_envio', 'hora_envio', 'msg_id','arquivo'])
        writer.writerow([remetente, assunto, data_envio, hora_envio, msg_id, nome_arquivo])

    print(f"📄 Encaminhamento registrado: {assunto} de {remetente} ({nome_arquivo})")

   
##########################################################################################
def normalizar_assunto(assunto):
    assunto = assunto.lower()
    assunto = re.sub(r'^(re|res|enc|fwd|fw)\s*:\s*', '', assunto)
    return assunto.strip()
    

def limpar_cid(texto):
    # Remove tags do tipo [cid:qualquer_coisa]
    return re.sub(r'\[cid:[^\]]+\]', '', texto)
#######################
def carregar_msg_ids(path_csv):
    try:
        df = pd.read_csv(path_csv, sep=';')
        ids = set(df['msg_id'].dropna().astype(str)) if 'msg_id' in df else set()
        arquivos = set(df['arquivo'].dropna().astype(str).str.lower()) if 'arquivo' in df else set()
        return ids, arquivos
    except FileNotFoundError:
        return set(), set()

msg_ids_registrados, arquivos_registrados = carregar_msg_ids(path_csv)

##############################
def tem_pdf(msg, arquivos_registrados):
    """
    Retorna True se houver pelo menos um anexo .pdf no e-mail.
    """
    for anexo in msg.attachments:
        filename = anexo.filename or ''
        if filename in ('3.Sgt PM Kelson e Sgt PM Marilson 204.2025.sec_0001.pdf', 'Oficio_00126551581.pdf',
                        'Extrato_00122965493_2VSJ_SGT_SANTOS.pdf','ADILSON MOREIRA_0001.pdf','Oficio_00124493202.pdf',
                        'PROCESSO_ 0000244-56.2025.8.17.3120 - CARTA PRECATÓRIA CÍVEL - 0000244-56.2025.8.17.3120-1762177111273-864398-processo.pdf',
                        'Requerimento_0087935422_EMAIL_1.pdf',
                        'Oficio_00122965336.pdf', 'Extrato_00122965493_2VSJ_SGT_SANTOS.pdf',
                        ):
            continue
        
        if filename.endswith('.pdf') and filename not in arquivos_registrados:
            return True
    return False
#########################################################
def tem_anexo_relevante(msg):
    for anexo in msg.attachments:
        filename = anexo.filename or ''
        if any(filename.lower().endswith(ext) for ext in ['.pdf', '.doc', '.docx']):#, '.jpg', '.png']):
            return True
    return False
# # Carrega seus dados CSV, monta um dicionário de assuntos normalizados
assuntos_map = {}

with open(path_csv, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter=';')
    dados = list(reader)
    # Criação de mapa de assuntos com vínculo direto aos dados
assuntos_map = {
    normalizar_assunto(row['assunto']): row 
    for row in dados
    if row['assunto']# ignora None ou vazio
}
#####################################################################################
def processar_encaminhamento(msg, msg_ids_registrados, arquivos_registrados):
    msg_id = str(getattr(msg, 'uid', '') or getattr(msg, 'message_id', '')).strip()

    if msg_id in msg_ids_registrados:
        return False

    if not tem_pdf(msg, arquivos_registrados):
        return False

    encaminhar_email_completo(msg)
    registrar_encaminhamento_por_assunto(msg)

    msg_ids_registrados.add(msg_id)
    return True

#####################################################################
def processar_resposta_projudi(msg, assuntos_map):
    assunto_msg = normalizar_assunto(msg.subject or "")
    if assunto_msg not in assuntos_map:
        return False

    row = assuntos_map[assunto_msg]

    if row.get('registrar_envio') in ('recebido', 'cumprido', 'informado', 'devolvido', 'juntado'):
        return False

    texto = msg.text or (msg.html and BeautifulSoup(msg.html, 'html.parser').get_text()) or ''
    texto = limpar_cid(texto)
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto[:100]

    data = msg.date.strftime('%d/%m/%Y')
    hora = msg.date.strftime('%H:%M:%S')

    assunto_para_juntada = f"{row['assunto'].split('Referente')[0].strip()} - {texto.strip()}"

    juntada_resposta_projudi(
        data_resposta=data,
        hora_resposta=hora,
        link_base=link_base,
        destinatario=msg.from_,
        url_recebimento=row['url_recebimento'],
        assunto=assunto_para_juntada,
        cookies=cookies,
        cookie_domain=cookie_domain,
        digitar=digitar,
        tempo_espera=tempo_espera
    )

    # 🔴 REGISTRO FINAL (em memória)
    row['status'] = 'recebido'
    row['resposta'] = texto
    row['registrar_envio'] = 'cumprido'

    return True

###################################################################
        
# for assunto, url_juntada in zip_longest(assuntos_map, url_recibo):
with MailBox(IMAP_SERVER).login(usuario, senha_app, initial_folder='INBOX') as mailbox:
    mensagens = mailbox.fetch(criteria=A(all=True), reverse=True, limit=100)
    for msg in mensagens:
        # Fluxo A – reencaminhamento
        processar_encaminhamento(msg, msg_ids_registrados, arquivos_registrados)
        # Fluxo B – resposta Projudi
        processar_resposta_projudi(msg, assuntos_map)
with open(path_csv, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=CAMPOS, delimiter=';')
    writer.writeheader()
    writer.writerows(dados)
    print(f"[✓] CSV atualizado com {len(dados)} registros.")